# Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os
import json

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import (
    classification_report, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
print("✅ Library siap")

✅ Library siap


d:\Tugas Besar ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Dataset

In [ ]:
train_df = pd.read_csv('../data/aug_train.csv')
test_df  = pd.read_csv('../data/aug_test.csv')

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
train_df.head()

# EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_df['target'].value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue','tomato'], edgecolor='black'
)
axes[0].set_title('Target Distribution')
axes[0].set_xticklabels(['Stay (0)', 'Move (1)'], rotation=0)

missing = train_df.isnull().sum()
missing[missing > 0].sort_values().plot(kind='barh', ax=axes[1], color='salmon')
axes[1].set_title('Missing Values per Column')

plt.tight_layout()
plt.show()

print("\nTarget ratio:\n", train_df['target'].value_counts(normalize=True).round(3))

# Analisis Missing Values

In [ ]:
missing_cols = ['gender', 'enrolled_university', 'education_level',
                'major_discipline', 'experience', 'company_size',
                'company_type', 'last_new_job']

for col in missing_cols:
    missing_mask = train_df[col].isnull()
    if missing_mask.sum() > 0:
        target_mean_missing     = train_df[missing_mask]['target'].mean()
        target_mean_not_missing = train_df[~missing_mask]['target'].mean()
        print(f"{col:20s} | missing→target={target_mean_missing:.3f} | "
              f"not missing→target={target_mean_not_missing:.3f} | "
              f"diff={abs(target_mean_missing - target_mean_not_missing):.3f}")

# Preprocessing dan Fitur Engineering

In [ ]:
def preprocess_v3_fe(df, is_train=True):
    df = df.copy()
    df.drop(columns=['enrollee_id', 'city'], inplace=True)

    if is_train:
        y = df.pop('target').astype(int)

    # 1. Missing Indicator Flags
    flag_cols = ['gender', 'company_size', 'company_type',
                 'major_discipline', 'last_new_job', 'enrolled_university']
    for col in flag_cols:
        df[f'{col}_missing'] = df[col].isnull().astype(int)

    # 2. Imputasi
    mode_cols = ['gender', 'enrolled_university', 'education_level',
                 'major_discipline', 'experience', 'last_new_job']
    for col in mode_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
    df['company_size'] = df['company_size'].fillna('Unknown')
    df['company_type'] = df['company_type'].fillna('Unknown')

    # 3. Feature Engineering
    df['job_hopper'] = (
        (df['last_new_job'].isin(['1', 'never'])) &
        (df['experience'].isin(['<1','1','2','3','4','5']))
    ).astype(int)

    df['high_value_candidate'] = (
        (df['experience'].isin(['10','11','12','13','14','15','>20'])) &
        (df['education_level'].isin(['Masters', 'Phd']))
    ).astype(int)

    df['is_fresher'] = (
        (df['experience'].isin(['<1', '1', '2'])) &
        (df['enrolled_university'] != 'no_enrollment')
    ).astype(int)

    df['mismatch_candidate'] = (
        (df['relevent_experience'] == 'No relevent experience') &
        (df['company_size'].isin(['<10', '10/49', '50-99', 'Unknown']))
    ).astype(int)

    df['ambitious'] = (
        (df['city_development_index'] < 0.75) &
        (df['training_hours'] > 50)
    ).astype(int)

    df['career_stagnant'] = (
        (df['last_new_job'].isin(['>4', '4'])) &
        (df['experience'].isin(['10','11','12','13','14','15','>20']))
    ).astype(int)

    exp_map = {'<1': 0.5, '>20': 21}
    exp_numeric = df['experience'].replace(exp_map)
    exp_numeric = pd.to_numeric(exp_numeric, errors='coerce').fillna(1)
    df['training_per_exp'] = df['training_hours'] / (exp_numeric + 1)

    df['city_tier'] = pd.cut(
        df['city_development_index'],
        bins=[0, 0.62, 0.78, 0.90, 1.0],
        labels=[0, 1, 2, 3]
    ).astype(int)

    # 4. Ordinal Encoding
    ordinal_config = {
        'experience'     : ['<1','1','2','3','4','5','6','7','8','9','10',
                            '11','12','13','14','15','16','17','18','19','20','>20'],
        'last_new_job'   : ['never','1','2','3','4','>4'],
        'company_size'   : ['<10','10/49','50-99','100-500','500-999',
                            '1000-4999','5000-9999','10000+','Unknown'],
        'education_level': ['Primary School','High School','Graduate','Masters','Phd'],
    }
    for col, order in ordinal_config.items():
        enc = OrdinalEncoder(
            categories=[order],
            handle_unknown='use_encoded_value',
            unknown_value=-1
        )
        df[col] = enc.fit_transform(df[[col]])

    # 5. Nominal Encoding
    nominal_cols = ['gender', 'relevent_experience', 'enrolled_university',
                    'major_discipline', 'company_type']
    for col in nominal_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    if is_train:
        return df, y
    return df


X, y = preprocess_v3_fe(train_df, is_train=True)
X_test_final = preprocess_v3_fe(test_df, is_train=False)

print(f"✅ Total fitur: {X.shape[1]} kolom")
print("Kolom:", X.columns.tolist())

# Training dan SMOTE

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("Sebelum SMOTE:", y_train.value_counts().to_dict())
print("Sesudah SMOTE:", pd.Series(y_train_res).value_counts().to_dict())

# Baseline Model

In [ ]:
base_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
base_rf.fit(X_train_res, y_train_res)

y_pred_base  = base_rf.predict(X_val)
y_proba_base = base_rf.predict_proba(X_val)[:, 1]

print("=== BASELINE RANDOM FOREST ===")
print(classification_report(y_val, y_pred_base))
print(f"ROC-AUC : {roc_auc_score(y_val, y_proba_base):.4f}")

# Feature Importance

In [ ]:
importances = pd.Series(base_rf.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 7))
importances_sorted.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importances - Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print(f"\nTop 10 fitur:")
print(importances.sort_values(ascending=False).head(10))

# Hyperparameter Tuning

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 200, 700),
        'max_depth'        : trial.suggest_int('max_depth', 8, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 15),
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 2, 6),
        'max_features'     : trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'class_weight'     : trial.suggest_categorical('class_weight', ['balanced', None]),
    }
    rf = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    score = cross_val_score(rf, X_train_res, y_train_res,
                            cv=cv, scoring='roc_auc', n_jobs=-1).mean()
    return score

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n✅ Best ROC-AUC (CV): {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

# Train Model Final

In [ ]:
best_rf = RandomForestClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)
best_rf.fit(X_train_res, y_train_res)
print("✅ Model final selesai ditraining")

# Evaluasi Model

In [ ]:
y_pred  = best_rf.predict(X_val)
y_proba = best_rf.predict_proba(X_val)[:, 1]

print("=== FINAL RANDOM FOREST ===")
print(classification_report(y_val, y_pred))
print(f"ROC-AUC : {roc_auc_score(y_val, y_proba):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(y_val, y_pred, ax=axes[0])
axes[0].set_title('Confusion Matrix')
RocCurveDisplay.from_predictions(y_val, y_proba, ax=axes[1])
axes[1].set_title('ROC Curve')
plt.tight_layout()
plt.show()

print("\n=== PERBANDINGAN ===")
print(f"Baseline AUC : {roc_auc_score(y_val, y_proba_base):.4f}")
print(f"Tuned AUC    : {roc_auc_score(y_val, y_proba):.4f}")

# Optimasi threshold 

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

# Cari threshold terbaik
y_proba = best_rf.predict_proba(X_val)[:, 1]
thresholds = np.arange(0.1, 0.9, 0.01)

best_thresh = 0.5
best_f1     = 0

for thresh in thresholds:
    y_pred_t = (y_proba >= thresh).astype(int)
    f1 = f1_score(y_val, y_pred_t, average='macro')
    if f1 > best_f1:
        best_f1     = f1
        best_thresh = thresh

print(f"Threshold default (0.5) → Macro F1: {f1_score(y_val, best_rf.predict(X_val), average='macro'):.4f}")
print(f"Threshold terbaik ({best_thresh:.2f}) → Macro F1: {best_f1:.4f}")

In [ ]:
y_pred_optimized = (y_proba >= best_thresh).astype(int)

print("=== DENGAN THRESHOLD OPTIMAL ===")
print(classification_report(y_val, y_pred_optimized))
print(f"ROC-AUC : {roc_auc_score(y_val, y_proba):.4f}")

# Simpan Model

In [ ]:
os.makedirs('../models', exist_ok=True)

joblib.dump(best_rf, '../models/random_forest_model.pkl')

feature_names = X.columns.tolist()
joblib.dump(feature_names, '../models/selected_features.pkl')

with open('../models/best_params_rf.json', 'w') as f:
    json.dump(study.best_params, f, indent=2)

joblib.dump({
    'n_features'   : X.shape[1],
    'feature_names': feature_names,
    'model_version': 'FE+Tuned',
    'auc'          : round(roc_auc_score(y_val, y_proba), 4),
    'macro_f1'     : round(best_f1, 4),
    'threshold'    : round(best_thresh, 2)
}, '../models/model_metadata.pkl')

print("✅ Model final tersimpan!")
print(f"Threshold optimal : {best_thresh:.2f}")
print(f"Macro F1 optimal  : {best_f1:.4f}")
print("Files:", os.listdir('../models'))

# Sanity Cek

In [ ]:
sample = X_val.iloc[:5]
pred   = best_rf.predict(sample)
proba  = best_rf.predict_proba(sample)[:, 1]

result = pd.DataFrame({
    'Actual'      : y_val.iloc[:5].values,
    'Predicted'   : pred,
    'Prob(Pindah)': proba.round(3)
})
print(result)